In [ ]:

from pprint import pprint
import datasets as datasets_lib
import grain
import pandas as pd
import os
import fsspec
import numpy as np

import transformers
from tunix.generate import mappings

Dataset = datasets_lib.Dataset
AutoTokenizer = transformers.AutoTokenizer

from absl import logging as absl_logging

import logging
import sys

# ====== Logging Configuration ======
# 1. Force absl to use python logging
absl_logging.use_python_logging()

# 2. Configure the root logger
logging.basicConfig(
    stream=sys.stdout,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)

# 3. Explicitly set levels for relevant loggers
logging.getLogger().setLevel(logging.INFO)
logging.getLogger("absl").setLevel(logging.INFO)

# 4. Set absl verbosity
absl_logging.set_verbosity(absl_logging.INFO)
absl_logging.set_stderrthreshold("info")

print("Logging configured at INFO level.")
import os

os.environ["JAX_COMPILER_CACHE_TGTS"] = ""

In [ ]:
try:
  from GOOGLE_INTERNAL_PACKAGE_PATH.pyglib import gfile
  from etils import ecolab

  cm = ecolab.adhoc(
      source=ecolab.FROM_NOTEBOOK_OR_HEAD,
      reload="tunix",
      behavior="preferred",
      cell_autoreload=True,
  )

  file_open = gfile.Open

  NOTEBOOK_ENV = "g3"
except Exception:
  NOTEBOOK_ENV = "git"

  import contextlib
  cm = contextlib.nullcontext()

  file_open = fsspec.open


In [ ]:

with cm:
  from tunix.models.qwen2 import model as qwen2_lib
  from tunix.models.qwen2 import params as qwen2_params_lib
  from tunix.models.qwen3 import model as qwen3_model_lib
  from tunix.models.qwen3 import params as qwen3_params_lib
  from tunix.models.gemma4 import model as gemma4_lib
  from tunix.models.gemma4 import params_safetensors as gemma4_params_lib
  from tunix.generate import sampler as sampler_lib
  from tunix.utils import math_utils
# %%
from typing import Any, Dict, Optional
import jax
from jax import numpy as jnp
from flax import nnx
import orbax.checkpoint as ocp
from tqdm.auto import tqdm
import re

In [ ]:

MODEL_PATH_PREFIX = "gs://tunix/models"
MODEL_MAPPING = {
    "Qwen/Qwen2.5-1.5B-Instruct": (
        qwen2_lib.ModelConfig.qwen2p5_1p5b(),
        os.path.join(MODEL_PATH_PREFIX, "qwen2_5/torch/1.5b-it"),
    ),
    "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B": (
        qwen2_lib.ModelConfig.deepseek_r1_distill_qwen_1p5b(),
        os.path.join(MODEL_PATH_PREFIX, "DeepSeek-R1-Distill-Qwen-1.5B"),
    ),
    "agentica-org/DeepScaleR-1.5B-Preview": (
        qwen2_lib.ModelConfig.deepseek_r1_distill_qwen_1p5b(),
        os.path.join(MODEL_PATH_PREFIX, "DeepScaleR-1.5B-Preview"),
    ),
    "google/gemma-4-31B-it": (
      gemma4_lib.ModelConfig.gemma4_31b(),
      "/mnt/disks/linchai-data/huggingface/hub/models--google--gemma-4-31B-it/snapshots/439edf5652646a0d1bd8b46bfdc1d3645761a445",
    ),
    "google/gemma-4-E2B-it": (
      gemma4_lib.ModelConfig.gemma4_e2b(),
      "/mnt/disks/linchai-data/huggingface/hub/models--google--gemma-4-E2B-it/snapshots/b4a601102c3d45e2b7b50e2057a6d5ec8ed4adcf",
    ),
    "google/gemma-4-E4B-it": (
      gemma4_lib.ModelConfig.gemma4_e4b(),
      "/mnt/disks/linchai-data/huggingface/hub/models--google--gemma-4-E4B-it/snapshots/83df0a889143b1dbfc61b591bbc639540fd9ce4c",
    ),
    "google/gemma-4-26B-A4B-it": (
      gemma4_lib.ModelConfig.gemma4_26b_a4b(),
      "/mnt/disks/linchai-data/huggingface/hub/models--google--gemma-4-26B-A4B-it/snapshots/7d4c97e54145f8ffd1a4dd1b4986a5015a517842",
    ),
    "Qwen/Qwen3-1.7B": (
        qwen3_model_lib.ModelConfig.qwen3_1p7b(),
        "/home/linchai_google_com/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
    ),
    
}

In [ ]:
import os
os.environ["TPU_LOG_DIR"] = "~/my_tpu_logs"
os.environ["SKIP_JAX_PRECOMPILE"] = "True"

In [ ]:

# model_version = "google/gemma-4-26B-A4B-it"
# model_version = "google/gemma-4-31B-it"
model_version = "Qwen/Qwen3-1.7B"
model_config, model_path = MODEL_MAPPING[model_version]

tokenizer = AutoTokenizer.from_pretrained(model_version)

In [ ]:

trainer_mesh = jax.sharding.Mesh(np.array(jax.devices()).reshape(1, 8), axis_names=["fsdp", "tp"])
rollout_mesh = jax.sharding.Mesh(np.array(jax.devices()).reshape(1, 8), axis_names=["fsdp", "tp"])


In [ ]:

print("Loading model from safe tensors...")
with trainer_mesh:
  # trainer_model = gemma4_params_lib.create_model_from_safe_tensors(file_dir=model_path, config=model_config, mesh=trainer_mesh)
  trainer_model = qwen3_params_lib.create_model_from_safe_tensors(file_dir=model_path, config=model_config, mesh=trainer_mesh)
  ref_model = qwen3_params_lib.create_model_from_safe_tensors(file_dir=model_path, config=model_config, mesh=trainer_mesh)

import jax
jax.clear_caches()

In [ ]:
import optax
from tunix.rl import rl_cluster as rl_cluster_lib
from tunix.rl.rollout import base_rollout

optimizer = optax.adamw(
    learning_rate=1e-6,
    b1=0.9,
    b2=0.99,
    weight_decay=0.01,
)

base_rollout_dict = {
    "max_prompt_length": 32,
    "kv_cache_size": 32 + 16 + 256,
    "temperature": 0.7,
    "top_p": 0.95,
    "top_k": None,
    "return_logprobs": True,
    "max_tokens_to_generate": 16,
}

vllm_rollout_dict = {
    # vllm-tpu specific configs
    "rollout_vllm_model_version": model_version,
    "rollout_vllm_hbm_utilization": 0.7,
    "rollout_vllm_tpu_backend_type": "jax",
    "rollout_vllm_server_mode": True,
    "rollout_vllm_enable_dp_attention": True,
    "rollout_vllm_async_scheduling": True,
    "rollout_vllm_init_with_random_weights": False,
    "tensor_parallel_size": 8,
    "data_parallel_size": 1,
    "rollout_vllm_delete_dst_buffers": True,
    "rollout_vllm_max_num_seqs": 1,
    "rollout_vllm_max_num_batched_tokens": 2496,
    "rollout_vllm_kwargs": {
        "kv_cache_metrics": True,
        "disable_log_stats": False,
        "enable_prefix_caching": False,
    },
}

rollout_engine_config = base_rollout.RolloutConfig(
  **base_rollout_dict, **vllm_rollout_dict
)
  
cluster_config = rl_cluster_lib.ClusterConfig(
    role_to_mesh={
        rl_cluster_lib.Role.ACTOR: trainer_mesh,
        rl_cluster_lib.Role.REFERENCE: trainer_mesh,
        rl_cluster_lib.Role.ROLLOUT: rollout_mesh,
    },
    rollout_engine="vllm",
    offload_to_cpu=False,
    training_config=rl_cluster_lib.RLTrainingConfig(
        actor_optimizer=optimizer,
        eval_every_n_steps=1,
        max_steps=1,
        mini_batch_size=1,
        train_micro_batch_size=1,
    ),
    rollout_config=rollout_engine_config,
)
rl_cluster = rl_cluster_lib.RLCluster(
    actor=trainer_model,
    reference=trainer_model,
    tokenizer=tokenizer,
    cluster_config=cluster_config,
)

In [ ]:

prompts = [{"role": "user", "content": "which is bigger, 1 or 2"}]
prompts = tokenizer.apply_chat_template(prompts, add_generation_prompt=True, tokenize=False)
out_data = rl_cluster.generate(prompts=prompts, max_generation_steps=16)
print("Text from VLLM sampler: ", out_data.text)
print("logprobs from VLLM sampler: ", out_data.logprobs)
print("tokens from VLLM sampler: ", out_data.tokens)
print("left_padded_prompt_tokens from VLLM sampler: ", out_data.left_padded_prompt_tokens)

In [ ]:
out_tokens = out_data.tokens[0]


prompt_tokens = out_data.left_padded_prompt_tokens[0]
completion_tokens = out_tokens
print(f"{prompt_tokens=}")
print(f"{completion_tokens=}")
prompt_tokens = jnp.repeat(jnp.array([prompt_tokens]), 4, axis=0)
completion_tokens = jnp.repeat(jnp.array([completion_tokens]), 4, axis=0)


from tunix.rl import common
graphdef, state = nnx.split(trainer_model)

batch_size = prompt_tokens.shape[0]
if batch_size == 0:
  raise ValueError(
      "Cannot get reference log probabilities from an empty batch."
  )

from tunix.sft import sharding_utils
with trainer_mesh:
    # This assumes reference model shards same data sharding as actor, which
    # should be true as ref model and policy model shares same architecture.
    dest_prompt_tokens = sharding_utils.shard_input(
        prompt_tokens,
         ("fsdp",),
    )
    dest_completion_tokens = sharding_utils.shard_input(
        completion_tokens,
         ("fsdp",),
    )
    dest_completion_mask = None

ref_logps = common.compute_per_token_logps(
    graphdef,
    state,
    prompt_tokens=dest_prompt_tokens,
    completion_tokens=dest_completion_tokens,
    pad_id=tokenizer.pad_token_id,
    eos_id=tokenizer.eos_token_id,
    temperature = 0.7,
)

In [ ]:

out_tokens = out_data.tokens[0]


prompt_tokens = out_data.padded_prompt_tokens[0]
completion_tokens = out_tokens
print(f"{prompt_tokens=}")
print(f"{completion_tokens=}")
prompt_tokens = jnp.repeat(jnp.array([prompt_tokens]), 4, axis=0)
completion_tokens = jnp.repeat(jnp.array([completion_tokens]), 4, axis=0)


from tunix.rl import common
graphdef, state = nnx.split(trainer_model)

batch_size = prompt_tokens.shape[0]
if batch_size == 0:
  raise ValueError(
      "Cannot get reference log probabilities from an empty batch."
  )

from tunix.sft import sharding_utils
with trainer_mesh:
    # This assumes reference model shards same data sharding as actor, which
    # should be true as ref model and policy model shares same architecture.
    dest_prompt_tokens = sharding_utils.shard_input(
        prompt_tokens,
         ("fsdp",),
    )
    dest_completion_tokens = sharding_utils.shard_input(
        completion_tokens,
         ("fsdp",),
    )
    dest_completion_mask = None

ref_logps = common.compute_per_token_logps(
    graphdef,
    state,
    prompt_tokens=dest_prompt_tokens,
    completion_tokens=dest_completion_tokens,
    pad_id=tokenizer.pad_token_id,
    eos_id=tokenizer.eos_token_id,
    temperature = 0.7,
)

In [ ]:
import jax
jax.clear_caches()
jax.block_until_ready(ref_logps)
print(f"{ref_logps.sharding = }")
print(f"{ref_logps[0] = }")

# float32 [-1.6212287e-05, -4.2914307e-05,  0.0000000e+00, -1.2116224e-03,  0.0000000e+00,  0.0000000e+00, -1.1920902e-07, -5.9604508e-07]
# bf16 [-1.6212287e-05, -4.1364681e-05,  0.0000000e+00, -1.2746054e-03, 0.0000000e+00,  0.0000000e+00, -1.1920902e-07, -4.7683608e-07]
# vllm sampler: [-1.8954058759845793e-05, -3.337797534186393e-05, 0.0, -0.0013262758729979396, 0.0, 0.0, -1.1920901954454166e-07, -4.7683607817816664e-07]

In [ ]:
logprobs_array = jnp.array(out_data.logprobs)
diff_mean = jnp.mean(jnp.abs(ref_logps - logprobs_array))
diff_std = jnp.std(jnp.abs(ref_logps - logprobs_array))
print(f"Mean absolute difference between reference log probabilities and VLLM sampler log probabilities: {diff_mean}")
print(f"Standard deviation of absolute difference: {diff_std}")